# Eigenmode / linear-stability analysis

This notebook demonstrates Nefes's **linear stability** capability: finding a network's
free acoustic oscillations as the roots of the nonlinear eigenproblem

$$\det A(\omega) = 0,\qquad \omega = \omega_r + i\,\omega_i \in \mathbb{C},$$

where $A(\omega)$ is the *same* assembled perturbation operator the forced-response
driver uses. The roots are found by **Beyn's contour-integral method** over a region of
the complex plane, with an **argument-principle completeness certificate** that counts
the modes a region truly contains and re-searches until they are all resolved.

**Reading the output**

* modal frequency $f = \omega_r / 2\pi$ (Hz),
* growth rate $\sigma = -\,\mathrm{Im}(\omega)$ (1/s) — a mode is **unstable** when $\sigma > 0$;
  every passive network below is stable ($\sigma \le 0$). Active (thermoacoustic) growth
  needs a heat-release source, which is planned.
* `res.certified` / `res.expected` report the completeness certificate.

The cases build up from a textbook organ pipe (validated against closed form) to an
area-change resonator and a larger multi-segment network.

In [ ]:
import time

import numpy as np
import plotly.graph_objects as go

from nefes.shell import Network
from nefes.elements import catalog as cat
from nefes.thermo.configure import perfect_gas
from nefes.assembly.recover import ES_U, ES_C
from nefes.perturbation import PerturbationBC, eigenmodes
from nefes.plotting.theme import NEFES_TEMPLATE_NAME

GAS = perfect_gas(287.0, 1.4)  # air: R = 287 J/kg/K, gamma = 1.4


def mean_uc(sol, edge=0):
    """Mean axial velocity and sound speed on an edge of a converged solution."""
    t = sol.table()
    return float(t[ES_U, edge]), float(t[ES_C, edge])

## 1. Organ pipe — validation against $n\,c/2L$

The simplest case: a single rigid-walled (closed-closed) duct. With no mean flow and no
losses, the acoustic modes are the textbook organ-pipe set $f_n = n\,c/2L$, all
*marginally* stable ($\sigma \approx 0$). This pins the solver against closed form.

In [ ]:
L = 0.5  # duct length [m]

net = Network(GAS, p_ref=101325.0, T_ref=300.0, mdot_ref=5.0)
net.add(cat.total_pressure_inlet(101325.0, 300.0, perturbation_bc=PerturbationBC.hard_wall()))
net.add(cat.duct(L))
net.add(cat.wall())  # rigid wall at both ends -> quiescent, lossless
net.connect(0, 1, 0.05)
net.connect(1, 2, 0.05)
sol = net.solve()
assert sol.converged

u, c = mean_uc(sol)
f1 = c / (2 * L)
res = eigenmodes(sol.problem, sol.x, (0.5 * f1, 4.5 * f1))

print(f"sound speed c = {c:.1f} m/s,  mean flow u = {u:.2e} m/s (quiescent)")
print(f"certified complete: {res.certified}  "
      f"(found {res.n_modes}, argument-principle count {res.expected})\n")
print(f"{'mode':>4} {'f [Hz]':>10} {'n c/2L [Hz]':>12} {'growth [1/s]':>13} {'residual':>10}")
for i, m in enumerate(res.summary(), start=1):
    print(f"{i:>4} {m['freq_hz']:>10.2f} {i * f1:>12.2f} "
          f"{m['growth_rate']:>13.2e} {m['residual']:>10.1e}")

In [ ]:
res.plot_spectrum(title="Organ pipe (rigid-rigid): marginally-stable acoustic modes").show()

## 2. Boundary conditions set both frequency *and* stability

The terminal `PerturbationBC`s decide the spectrum. Three variants of the same duct:

* **closed-closed** (rigid-rigid): $f_n = n\,c/2L$;
* **open-closed**: a quarter-wave resonator, $f = (2k{+}1)\,c/4L$;
* **rigid + absorbing outlet** ($|R| = 0.6$): the same frequencies as closed-closed but
  now every mode **decays** — the passive-damping signature on the growth-rate axis.

In [ ]:
def duct_with_bcs(inlet_bc, outlet_elem, L=0.5):
    net = Network(GAS, p_ref=101325.0, T_ref=300.0, mdot_ref=5.0)
    net.add(cat.total_pressure_inlet(101325.0, 300.0, perturbation_bc=inlet_bc))
    net.add(cat.duct(L))
    net.add(outlet_elem)
    net.connect(0, 1, 0.05)
    net.connect(1, 2, 0.05)
    s = net.solve()
    assert s.converged
    return s


cases = {
    "closed-closed (rigid-rigid)": (PerturbationBC.hard_wall(), cat.wall()),
    "open-closed (quarter-wave)": (PerturbationBC.open_end(), cat.wall()),
    "rigid + absorbing R=0.6": (
        PerturbationBC.hard_wall(),
        cat.pressure_outlet(101325.0, 300.0, perturbation_bc=PerturbationBC.reflection(0.6)),
    ),
}

band = (100.0, 1100.0)  # Hz; holds three modes of each case, clear of every mode edge
results = {}
for label, (ibc, oel) in cases.items():
    s = duct_with_bcs(ibc, oel)
    r = eigenmodes(s.problem, s.x, band)
    results[label] = r
    print(f"{label:<28} freqs={np.round(r.freqs, 1)}  "
          f"growth={np.round(r.growth_rates, 1)}  certified={r.certified}")

In [ ]:
results["rigid + absorbing R=0.6"].plot_spectrum(
    title="Absorbing outlet (|R| = 0.6): every mode decays (growth < 0)"
).show()

## 3. Mean flow — convective shift of the modes

With subsonic mean flow the fundamental drops to $f_1 = \dfrac{c}{2L}\,(1 - M^2)$ as the
up/downstream wave transit times become unequal. Driving a range of inlet total
pressures sweeps the Mach number; the eigenmode fundamental tracks the $1 - M^2$ law.

In [ ]:
L, R = 0.5, 0.7
rows = []
for pt in (104000.0, 115000.0, 128000.0, 143000.0):
    net = Network(GAS, p_ref=101325.0, T_ref=300.0, mdot_ref=5.0)
    net.add(cat.total_pressure_inlet(pt, 300.0, perturbation_bc=PerturbationBC.reflection(R)))
    net.add(cat.duct(L))
    net.add(cat.pressure_outlet(101325.0, 300.0, perturbation_bc=PerturbationBC.reflection(R)))
    net.connect(0, 1, 0.05)
    net.connect(1, 2, 0.05)
    s = net.solve()
    assert s.converged
    u, c = mean_uc(s)
    M = u / c
    f0 = c / (2 * L)
    r = eigenmodes(s.problem, s.x, (0.25 * f0, 1.3 * f0))
    f1 = r.freqs[np.argmin(np.abs(r.freqs - f0 * (1 - M**2)))]
    rows.append((M, f1, f0))
    print(f"M = {M:.3f}   f1 found = {f1:7.2f} Hz   c/2L (1 - M^2) = {f0 * (1 - M**2):7.2f} Hz")

In [ ]:
M = np.array([r[0] for r in rows])
f1 = np.array([r[1] for r in rows])
f0 = np.array([r[2] for r in rows])
mm = np.linspace(0.0, M.max() * 1.05, 100)

fig = go.Figure()
fig.add_trace(go.Scatter(x=mm, y=1 - mm**2, mode="lines", name="1 - M²  (theory)"))
fig.add_trace(go.Scatter(x=M, y=f1 / f0, mode="markers", name="eigenmode  f₁(M)/f₁(0)",
                         marker=dict(size=12)))
fig.update_layout(template=NEFES_TEMPLATE_NAME, title="Convective shift of the fundamental",
                  xaxis_title="mean-flow Mach number M", yaxis_title="f₁(M) / f₁(0)")
fig.show()

## 4. Element combo — expansion chamber (area-change detuning)

A pipe → chamber → pipe network (two area changes) is the classic acoustic filter. Kept
quiescent (rigid both ends) so only the acoustic impedance jump matters. At **area ratio
1** it collapses to a uniform pipe and recovers the organ-pipe ladder; a larger ratio
**detunes** the modes — the standing waves rearrange around the chamber. The completeness
certificate holds throughout. (At $u = 0$ an isentropic and a sudden area change impose
the same junction condition, so they give identical acoustics here.)

In [ ]:
def quiescent_chamber(area_ratio, L_pipe=0.3, L_chamber=0.4):
    A1, A2 = 0.01, 0.01 * area_ratio
    net = Network(GAS, p_ref=101325.0, T_ref=300.0, mdot_ref=2.0)
    net.add(cat.total_pressure_inlet(101325.0, 300.0, perturbation_bc=PerturbationBC.hard_wall()))
    net.add(cat.duct(L_pipe))  # inlet pipe
    net.add(cat.isentropic_area_change("expansion"))
    net.add(cat.duct(L_chamber))  # chamber
    net.add(cat.isentropic_area_change("contraction"))
    net.add(cat.duct(L_pipe))  # outlet pipe
    net.add(cat.wall())  # rigid both ends -> quiescent
    net.connect(0, 1, A1)
    net.connect(1, 2, A1)
    net.connect(2, 3, A2)
    net.connect(3, 4, A2)
    net.connect(4, 5, A1)
    net.connect(5, 6, A1)
    s = net.solve()
    assert s.converged
    return s


L_tot = 0.3 + 0.4 + 0.3
band = (50.0, 950.0)
for ar in (1.0, 9.0):
    s = quiescent_chamber(ar)
    r = eigenmodes(s.problem, s.x, band)
    tag = "uniform pipe" if ar == 1.0 else f"area ratio {ar:.0f}"
    print(f"{tag:<16} freqs = {np.round(r.freqs, 1)}  certified = {r.certified}")

c = mean_uc(quiescent_chamber(1.0))[1]
print(f"\nuniform-pipe organ modes n·c/2L = "
      f"{[round(n * c / (2 * L_tot), 1) for n in range(1, 6)]} Hz")

## 5. Isentropic option — standard acoustic analysis

Classical acoustics usually assumes **isentropic** perturbations, $\rho' = p'/c^2$, dropping
the convected entropy wave. Passing `isentropic=True` pins the entropy characteristic
$h = \rho' - p'/c^2$ to zero on every edge — reusing the *same* operator, solver, and
certificate (no reconfiguration). It is especially useful when mean flow crosses area
changes: those inject entropy/convective modes that crowd the spectrum, which the isentropic
option removes, leaving the pure acoustic modes.

Below: the expansion chamber **with mean flow**. The full three-wave model returns the
acoustic modes *plus* many entropy/convective ones; `isentropic=True` returns only the
acoustic spectrum, with the entropy content of every mode zero to machine precision.

In [ ]:
def flowing_chamber(L_pipe=0.3, L_chamber=0.4):
    A1, A2 = 0.01, 0.05
    net = Network(GAS, p_ref=101325.0, T_ref=300.0, mdot_ref=2.0)
    net.add(cat.total_pressure_inlet(112000.0, 300.0, perturbation_bc=PerturbationBC.hard_wall()))
    net.add(cat.duct(L_pipe))
    net.add(cat.sudden_area_change("expansion"))  # mean flow -> entropy generated here
    net.add(cat.duct(L_chamber))
    net.add(cat.sudden_area_change("contraction"))
    net.add(cat.duct(L_pipe))
    net.add(cat.pressure_outlet(101325.0, 300.0, perturbation_bc=PerturbationBC.reflection(0.5)))
    for a, b, area in [(0, 1, A1), (1, 2, A1), (2, 3, A2), (3, 4, A2), (4, 5, A1), (5, 6, A1)]:
        net.connect(a, b, area)
    s = net.solve()
    assert s.converged
    return s


sol = flowing_chamber()
u, c = mean_uc(sol)
band = (60.0, 600.0)
full = eigenmodes(sol.problem, sol.x, band)
isen = eigenmodes(sol.problem, sol.x, band, isentropic=True)

hmax = max(np.max(np.abs(isen.mode_shape(i, basis="char")[:, 2])) for i in range(isen.n_modes))
print(f"inlet-pipe mean Mach            = {u / c:.3f}")
print(f"full 3-wave model               = {full.n_modes} modes (acoustic + entropy/convective)")
print(f"isentropic (rho' = p'/c^2)      = {isen.n_modes} modes (acoustic only), certified={isen.certified}")
print(f"isentropic acoustic freqs       = {np.round(isen.freqs, 1)} Hz")
print(f"max entropy amplitude (all modes) = {hmax:.1e}  (zero -> isentropic enforced)")

## 6. Scaling to a larger network

A long duct built from many segments in series — a genuinely larger operator — yet with a
known answer: the global modes are still $n\,c/2L_{\text{tot}}$. This exercises the band
sub-tiling and the completeness certificate at scale, and (with many edges) gives a
spatially-resolved **mode shape** along the pipe.

In [ ]:
def segmented_duct(n_seg, L_seg=0.075):
    net = Network(GAS, p_ref=101325.0, T_ref=300.0, mdot_ref=5.0)
    net.add(cat.total_pressure_inlet(101325.0, 300.0, perturbation_bc=PerturbationBC.hard_wall()))
    for _ in range(n_seg):
        net.add(cat.duct(L_seg))
    net.add(cat.wall())
    n_nodes = n_seg + 2
    for i in range(n_nodes - 1):
        net.connect(i, i + 1, 0.05)
    s = net.solve()
    assert s.converged
    return s, n_seg * L_seg


n_seg = 40
sol, L_tot = segmented_duct(n_seg)
u, c = mean_uc(sol)
f1 = c / (2 * L_tot)

t0 = time.perf_counter()
res = eigenmodes(sol.problem, sol.x, (0.5 * f1, 10.5 * f1))
dt = time.perf_counter() - t0

n = res.n_modes
err = np.max(np.abs(np.sort(res.freqs) - f1 * np.arange(1, n + 1)))
print(f"segments              = {n_seg}")
print(f"operator size N       = {sol.problem.n_col}")
print(f"total length          = {L_tot:.2f} m,   f1 = c/2L = {f1:.2f} Hz")
print(f"modes found           = {n} in {dt:.2f} s   (certified complete: {res.certified})")
print(f"max |f - n c/2L|      = {err:.3f} Hz")

In [ ]:
res.plot_spectrum(
    title=f"{n_seg}-segment duct: {res.n_modes} modes resolved (N = {sol.problem.n_col})"
).show()
res.plot_mode(2, basis="primitive",
              title="Mode 3 shape along the duct (primitive variables)").show()

## Takeaways

* The eigenmode driver mirrors the forced-response workflow: take a converged `Solution`,
  call `eigenmodes(prob, x, freq_band)`, read modal frequencies (Hz), growth rates, and
  mode shapes off the `EigenmodeResult`.
* Validated against closed form (organ pipe, quarter-wave, $1 - M^2$ convective shift) and
  shown on richer configurations (area-change resonator, multi-segment network).
* Every result is **completeness-certified** by the argument principle — `res.certified`
  guarantees no mode in the region was missed, with no hand-tuning.
* `isentropic=True` gives a pure acoustic analysis ($\rho' = p'/c^2$) on the same operator
  and solver, dropping the entropy/convective modes — ideal when mean flow crosses area
  changes.
* All cases here are passive and therefore stable ($\sigma \le 0$). Lighting up *growing*
  (thermoacoustic) modes needs an active heat-release source $S(\omega)$ — the Beyn driver
  already does not depend on the source type, so that element drops straight into this same spectrum.

## Export for FNetLibUI

The full network is available as `network` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.to_yaml(path)` embeds the mean-flow result fields, `network.to_yaml(path)` writes the topology only — then open the file in FNetLibUI.

In [ ]:
import os, tempfile

network, sol = sol.network, sol  # the primary Network and its mean-flow Solution
_out = os.path.join(tempfile.mkdtemp(), "eigenmode_analysis.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use network.to_yaml(_out) for topology only
print("saved case:", _out)